In [19]:
"""
Pentacam CSV Consolidator
==========================

What this does
---------------
Folder layout expected:

    C:\\HYD Extracted Pentacam\\
        Adarsh School\\
            BADisplay-LOAD_01-09-2025.csv
            BADisplay-LOAD_02-09-2025.csv
            BADisplay-LOAD_03-09-2025.csv
            INDEX-LOAD_01-09-2025.csv
            ...
        Another School\\
            ...

For each school folder, this script:
  1. Groups CSVs by "base name" (e.g. BADisplay-LOAD), ignoring the date in the filename.
  2. Reads each CSV, auto-detecting the delimiter (does the job of Excel's
     "Text to Columns" automatically -- no manual step needed).
  3. Concatenates all dated files of the same base name into one table.
  4. Extracts the ID columns + the specific measurement columns you need from
     each base-name table.
  5. Merges the 5 base-name tables together per patient/eye, using the ID
     columns as the join key (so one row = one patient exam, with every
     requested measurement in it).
  6. Writes one Excel file per school into "Consolidated Reports", plus one
     combined workbook with every school stacked together.

How to run
----------
1. Install dependencies (once):
       pip install pandas openpyxl

2. Edit ROOT_FOLDER below if your path is different.

3. Run:
       python pentacam_consolidate.py

4. Check the "Consolidated Reports" folder created inside ROOT_FOLDER.

If a school's output looks wrong (missing columns / rows not merging), the
most likely cause is that the actual filenames don't match the date pattern
this script expects, or a header spelling differs slightly from what's in
COLUMN_MAP below. See the troubleshooting notes at the bottom of this file.
"""

import os
import re
import sys
from pathlib import Path

import pandas as pd

# ---------------------------------------------------------------------------
# CONFIG -- edit these if needed
# ---------------------------------------------------------------------------

ROOT_FOLDER = r"C:\HYD Extracted Pentacam"
OUTPUT_SUBFOLDER = "Consolidated Reports"

# Columns that identify a patient/exam and are used to join the different
# file types together.
IDENTIFIER_COLUMNS = [
    "Last Name:",
    "First Name:",
    "Pat-ID:",
    "D.o.Birth:",
    "Exam Eye:",
]

# Base export file name (without date, without extension) -> list of exact
# column headers to pull out of that file.
COLUMN_MAP = {
    "INDEX-LOAD": [
        "K Max (Front):",
        "K Max X (Front):",
        "K Max Y (Front):",
    ],
    "CorneoScleral-LOAD": [
        "HWTW:",
    ],
    "CHAMBER-LOAD": [
        "C.Volume:",
        "C.Angle:",
        "Pupil:",
    ],
    "PACHY-LOAD": [
        "AC Depth",
    ],
    "BADisplay-LOAD": [
        "Pachy Prog Index Min.:",
        "Pachy Prog Index Max.:",
        "Pachy Prog Index Avg.:",
        "ART Max.:",
        "ART Min.:",
        "ART Avg.:",
        "Dist. Apex-Thin.Loc. [mm]:",
        "BAD Df:",
        "BAD Db:",
        "BAD Dp:",
        "BAD Dt:",
        "BAD Dam:",
        "BAD D:",
        "Status:"

    ],
}

# Regex used to strip a trailing date from a filename stem so that
# "BADisplay-LOAD_01-09-2025" -> "BADisplay-LOAD".
# Matches things like: _01-09-2025 / -01-09-2025 / 01.09.2025 / 01_09_2025
# with 2 or 4 digit years, at the end of the filename.
DATE_SUFFIX_RE = re.compile(
    r"[_\-\s]*\d{1,2}[-_.]\d{1,2}[-_.]\d{2,4}\s*$"
)


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def strip_date_suffix(stem: str) -> str:
    """Remove a trailing date pattern from a filename stem."""
    return DATE_SUFFIX_RE.sub("", stem).strip(" _-")


def read_csv_smart(path: Path) -> pd.DataFrame:
    """
    Read a Pentacam export CSV. Pentacam exports are semicolon-delimited, so
    that is tried first and explicitly -- letting pandas/csv.Sniffer
    auto-detect the delimiter is unreliable here because some header names
    contain a literal comma (e.g. "Chord Mu (polar vector, rel. to pupil
    center) [mm]:"), which can fool the sniffer into picking comma and
    mangling the whole file into one blob per row.
    """
    last_error = None
    for encoding in ("utf-8-sig", "latin1", "utf-16"):
        try:
            # 1. Try semicolon explicitly first -- this is the real Pentacam format.
            df = pd.read_csv(path, sep=";", engine="python", encoding=encoding, dtype=str)
            if df.shape[1] <= 1:
                # 2. Fall back to comma.
                df_comma = pd.read_csv(path, sep=",", engine="python", encoding=encoding, dtype=str)
                if df_comma.shape[1] > 1:
                    df = df_comma
                else:
                    # 3. Last resort: let pandas guess.
                    df = pd.read_csv(path, sep=None, engine="python", encoding=encoding, dtype=str)

            df.columns = [c.strip() for c in df.columns]
            return df
        except Exception as e:
            last_error = e
            continue
    raise RuntimeError(f"Could not read {path}: {last_error}")


def normalize(name: str) -> str:
    """Loose match key: lowercase, no spaces, no colon/punctuation quirks."""
    return re.sub(r"[^a-z0-9]", "", name.lower())


def find_column(df: pd.DataFrame, target: str) -> str | None:
    """
    Find target column in df.
      Tier 1: exact match.
      Tier 2: loose match ignoring case/spacing/punctuation.
      Tier 3: prefix match ignoring case/spacing/punctuation -- handles cases
              where the real export appends units or extra text, e.g. the
              dictionary says "K Max (Front):" but the real header is
              "K Max (Front) [D]:".
    """
    if target in df.columns:
        return target

    norm_target = normalize(target)

    for col in df.columns:
        if normalize(col) == norm_target:
            return col

    candidates = [col for col in df.columns if normalize(col).startswith(norm_target)]
    if len(candidates) == 1:
        return candidates[0]
    elif len(candidates) > 1:
        # Ambiguous -- prefer the shortest match (closest to the target itself)
        candidates.sort(key=lambda c: len(normalize(c)))
        print(f"    NOTE: '{target}' matched multiple columns via prefix match "
              f"{candidates} -- using '{candidates[0]}'. Verify this is correct.")
        return candidates[0]

    return None


def group_files_by_base_name(school_folder: Path):
    """Return dict: base_name -> list of csv file paths in this folder."""
    groups: dict[str, list[Path]] = {}
    for f in school_folder.iterdir():
        if f.is_file() and f.suffix.lower() == ".csv":
            base = strip_date_suffix(f.stem)
            groups.setdefault(base, []).append(f)
    return groups


def base_name_key(base_name: str) -> str | None:
    """Match a discovered base filename to a key in COLUMN_MAP, case-insensitively."""
    norm = normalize(base_name)
    for key in COLUMN_MAP:
        if normalize(key) == norm:
            return key
    return None


# ---------------------------------------------------------------------------
# Main processing
# ---------------------------------------------------------------------------

def process_school_folder(school_folder: Path) -> pd.DataFrame | None:
    print(f"\n--- Processing school: {school_folder.name} ---")
    groups = group_files_by_base_name(school_folder)

    if not groups:
        print("  No CSV files found, skipping.")
        return None

    per_file_tables = {}  # base_name -> extracted dataframe (IDs + target cols)

    for base_name, files in groups.items():
        mapped_key = base_name_key(base_name)
        if mapped_key is None:
            print(f"  Skipping '{base_name}' (not in COLUMN_MAP) -- files: {[f.name for f in files]}")
            continue

        print(f"  {mapped_key}: combining {len(files)} file(s) -> {[f.name for f in files]}")

        combined = []
        for f in files:
            try:
                df = read_csv_smart(f)
                df["__source_file__"] = f.name
                combined.append(df)
            except Exception as e:
                print(f"    WARNING: failed to read {f.name}: {e}")

        if not combined:
            continue

        full_df = pd.concat(combined, ignore_index=True, sort=False)

        # Resolve identifier columns actually present in this file
        id_cols_present = []
        missing_ids = []
        for id_col in IDENTIFIER_COLUMNS:
            found = find_column(full_df, id_col)
            if found:
                id_cols_present.append(found)
            else:
                missing_ids.append(id_col)

        if missing_ids:
            print(f"    WARNING: identifier column(s) not found in {mapped_key}: {missing_ids}")
            print(f"    Actual columns in this file were: {list(full_df.columns)}")

        if not id_cols_present:
            print(f"    ERROR: NONE of the identifier columns matched in {mapped_key} -- "
                  f"cannot merge this file with the others, skipping it.")
            continue

        # Resolve target columns
        target_cols_present = []
        missing_targets = []
        for target_col in COLUMN_MAP[mapped_key]:
            found = find_column(full_df, target_col)
            if found:
                target_cols_present.append(found)
            else:
                missing_targets.append(target_col)

        if missing_targets:
            print(f"    WARNING: column(s) not found in {mapped_key}: {missing_targets}")
            print(f"    Actual columns in this file were: {list(full_df.columns)}")

        keep_cols = id_cols_present + target_cols_present
        extracted = full_df[keep_cols].copy()

        # Rename identifier columns to a common canonical name so different
        # files can be merged on them even if casing/spacing differed slightly.
        rename_map = {found: canon for canon, found in zip(IDENTIFIER_COLUMNS, id_cols_present)}
        extracted.rename(columns=rename_map, inplace=True)

        # Strip stray whitespace from identifier VALUES (not just headers) --
        # e.g. "Left  " vs "Right" would otherwise silently break the merge.
        for canon in IDENTIFIER_COLUMNS:
            if canon in extracted.columns:
                extracted[canon] = extracted[canon].astype(str).str.strip()

        # Drop exact duplicate rows (in case a patient appears twice with
        # identical values across overlapping date exports)
        extracted.drop_duplicates(inplace=True)

        per_file_tables[mapped_key] = extracted

    if not per_file_tables:
        print("  Nothing extracted for this school.")
        return None

    # Merge all base-name tables together on the identifier columns
    merged = None
    for mapped_key, df in per_file_tables.items():
        if merged is None:
            merged = df
            continue

        # Join only on identifier columns present in BOTH tables so far.
        join_cols = [c for c in IDENTIFIER_COLUMNS if c in merged.columns and c in df.columns]

        if not join_cols:
            print(f"    ERROR: no common identifier columns between running merge and "
                  f"'{mapped_key}' -- appending its rows instead of merging (data will "
                  f"NOT line up per patient for this file; fix the identifier column "
                  f"names and re-run).")
            merged = pd.concat([merged, df], ignore_index=True, sort=False)
            continue

        merged = pd.merge(merged, df, on=join_cols, how="outer", suffixes=("", f"_{mapped_key}"))

    return merged


def main():
    root = Path(ROOT_FOLDER)
    if not root.exists():
        print(f"ERROR: root folder not found: {root}")
        sys.exit(1)

    output_dir = root / OUTPUT_SUBFOLDER
    output_dir.mkdir(exist_ok=True)

    all_schools = []

    for entry in sorted(root.iterdir()):
        if not entry.is_dir() or entry.name == OUTPUT_SUBFOLDER:
            continue

        result = process_school_folder(entry)
        if result is None or result.empty:
            continue

        out_path = output_dir / f"{entry.name}_Consolidated.xlsx"
        result.to_excel(out_path, index=False)
        print(f"  -> Saved {out_path}")

        result_with_school = result.copy()
        result_with_school.insert(0, "School", entry.name)
        all_schools.append(result_with_school)

    if all_schools:
        combined_all = pd.concat(all_schools, ignore_index=True, sort=False)
        combined_path = output_dir / "All_Schools_Combined.xlsx"
        combined_all.to_excel(combined_path, index=False)
        print(f"\nSaved combined workbook: {combined_path}")
    else:
        print("\nNo data was processed across any school folder.")


if __name__ == "__main__":
    main()

# ---------------------------------------------------------------------------
# Troubleshooting
# ---------------------------------------------------------------------------
# 1. "not in COLUMN_MAP" warnings for real Pentacam files:
#    Your actual filenames might strip/keep the date differently than assumed.
#    Print the school folder's file list and tell me the exact raw names --
#    I'll adjust DATE_SUFFIX_RE.
#
# 2. "identifier/column not found" warnings:
#    The header text in your real CSVs may differ slightly (extra space,
#    different colon, etc). The script already does a loose fallback match,
#    but if a column still isn't found, share one real header row and I'll
#    fix the exact string in COLUMN_MAP / IDENTIFIER_COLUMNS.
#
# 3. Rows not merging into one line per patient:
#    This happens if Pat-ID/Name/Exam Eye/D.o.Birth are formatted differently
#    across file types (e.g. one file exports date as "01.01.1990" and
#    another as "1990-01-01"). Let me know and I'll add normalization for
#    the specific field.
# ---------------------------------------------------------------------------


--- Processing school: ADARSHA VIDYALAYAM ---
  BADisplay-LOAD: combining 4 file(s) -> ['BADisplay-LOAD_01-09-2025.CSV', 'BADisplay-LOAD_02-09-2025.CSV', 'BADisplay-LOAD_03-09-2025.CSV', 'BADisplay-LOAD_06-11-2025.CSV']
  CHAMBER-LOAD: combining 4 file(s) -> ['CHAMBER-LOAD_01-09-2025.CSV', 'CHAMBER-LOAD_02-09-2025.CSV', 'CHAMBER-LOAD_03-09-2025.CSV', 'CHAMBER-LOAD_06-11-2025.CSV']
  Skipping 'COR-PWR-LOAD' (not in COLUMN_MAP) -- files: ['COR-PWR-LOAD_01-09-2025.CSV', 'COR-PWR-LOAD_02-09-2025.CSV', 'COR-PWR-LOAD_03-09-2025.CSV', 'COR-PWR-LOAD_06-11-2025.CSV']
  Skipping 'CorneaDens-LOAD' (not in COLUMN_MAP) -- files: ['CorneaDens-LOAD_01-09-2025.CSV', 'CorneaDens-LOAD_02-09-2025.CSV', 'CorneaDens-LOAD_03-09-2025.CSV', 'CorneaDens-LOAD_06-11-2025.CSV']
  CorneoScleral-LOAD: combining 4 file(s) -> ['CorneoScleral-LOAD_01-09-2025.CSV', 'CorneoScleral-LOAD_02-09-2025.CSV', 'CorneoScleral-LOAD_03-09-2025.CSV', 'CorneoScleral-LOAD_06-11-2025.CSV']
  Skipping 'EccSag-LOAD' (not in COLUMN_MAP)